In [1]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json

from textwrap import dedent

from pathlib import Path


from time import perf_counter

# Preliminari

In [3]:
# Configurazione OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_WITH_HINTS_FILE
TEMPERATURE = 0.0
BASELINE_INFERENCE = False
BASELINE_MODEL = constants.OPENAI_GPT_4_1_NANO

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, constants.AnnotationsExtended)

In [4]:
if False:
    print(system_prompt)

# Load Data

In [5]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {
    split: Path('../data/' + file_name) for split, file_name in file_names.items()
}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
    
print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 67
len(validation_data) = 61


# Generazione

## Preliminari generazione

In [6]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = constants.AnnotationsExtended):
    response = client.responses.parse(
        model=model,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=output_format
    )
    return response

In [8]:
# Esempio
if True:
    result = extract_data_from_report(BASELINE_MODEL, data['test'].iloc[0]['report_text'])

In [9]:
if False: # To see the full output
    pprint(result.model_dump())
if True:  # To see the string output text
    print(result.output_text)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.output_parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

{
  "morfologia": "solido_polipoide",
  "ore_inizio": null,
  "ore_fine": null,
  "spessore_parietale": null,
  "estensione_cranio_caudale": null,
  "distanza_oai": null,
  "posizione": {
    "basso": "si",
    "medio": "no",
    "alto": "no",
    "giunzione": "no"
  },
  "riflessione_peritoneale_anteriore": "non_valutabile",
  "infiltrazione_tessuto_adiposo": "no",
  "infiltrazione_sfinteri": "si",
  "infiltrazione_organi_extra": "si",
  "infiltrazione_organi_dettagli": {
    "pavimento_pelvico": "no",
    "altro": "si"
  },
  "coinvolgimento_riflessione_peritoneale": "no",
  "coinvolgimento_fascia_mesorettale": "si",
  "numero_linfonodi_non_conosciuto": "conosciuto",
  "linfonodi_sospetti": 1,
  "sedi_linfonodi": {
    "mesorettali": "no",
    "rettali_superiori": "no",
    "otturatori": "si",
    "iliaci": "no",
    "altro": "si"
  },
  "depositi_tumorali": "si",
  "emvi": "+",
  "stadio_T": "T4a",
  "stadio_N": "N2",
  "stadio_N1c": "no",
  "mrf": "-",
  "metastasi": "MX"
}


## Baseline inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [ ]:
if BASELINE_INFERENCE:
    risultati = []
    t1 = perf_counter()
    for i, testo in tqdm(enumerate(test_data['report_text'])):
        #print(f'*--- Processing report {i + 1}')
        output = extract_data_from_report(model=BASELINE_MODEL, report_text=testo)
        if output is None:
            risultati.append('no output')
        else:
            risultati.append(output.output_text)
    t2 = perf_counter()
    print(f'Tempo totale: {int((t2-t1)//60)} minuti {int((t2-t1))%60} secondi')

In [ ]:
if BASELINE_INFERENCE:
    results_dicts = []
    for r in risultati:
        results_dicts.append(
            {
                'model': BASELINE_MODEL,
                'prediction': json.loads(r)
            }
        )
    # Salvataggio
    with open(base_dir / 'data' / 'inference' / (BASELINE_MODEL + '.jsonl'), 'w', encoding='utf-8') as f:
        for r in results_dicts:
            f.write(json.dumps(r) + '\n')

## Finetune inference